#  추천 시스템 - High Cardinallity

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

import warnings
warnings.filterwarnings('ignore')

`(3) SQLite DB`

In [5]:
from langchain_community.utilities import SQLDatabase

# ETF 데이터베이스 연결
db = SQLDatabase.from_uri(
    "sqlite:///etf_database.db",
    )

# 사용 가능한 테이블 목록 
print(db.dialect)
print(db.get_usable_table_names())
etfs = db.run("SELECT * FROM ETFs LIMIT 5;")

for etf in eval(etfs):
    print(etf)

DatabaseError: (sqlite3.DatabaseError) file is not a database
[SQL: SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite~_%' ESCAPE '~' ORDER BY name]
(Background on this error at: https://sqlalche.me/e/20/4xp6)

## **고유명사 처리** 

- **고유명사** 처리를 위한 **벡터 스토어** 구축

- 사용자 질문 내 고유명사 **맞춤법 자동 검증** 기능

- 정확한 **엔티티 매칭**을 통한 쿼리 정확도 향상

- 벡터 기반 고유명사 자동 검증으로 검색 정확도 개선

- 참조: https://python.langchain.com/docs/tutorials/sql_qa/

### 1) **SQL QA Chain** 구현

In [ ]:
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langchain import hub
from langchain_community.tools import QuerySQLDatabaseTool
from langchain.prompts import ChatPromptTemplate
from langgraph.graph import START, StateGraph 
from IPython.display import Image, display

# 상태 정보를 저장하는 State 클래스
class State(TypedDict):
    question: str  # 입력 질문
    query: str     # 생성된 쿼리
    result: str    # 쿼리 결과
    answer: str    # 생성된 답변

# llm 모델 생성
llm = ChatOpenAI(model="gpt-4.1-mini")

# sql 쿼리 생성 프롬프트 템플릿 생성
query_prompt_template = ChatPromptTemplate.from_messages([
    ("system", """
    Given an input question, create a syntactically correct {dialect} query to run to help find the answer. 
    Unless the user specifies in his question a specific number of examples they wish to obtain, 
    always limit your query to at most {top_k} results.
      
    You can order the results by a relevant column to return the most interesting examples in the database.
    Never query for all the columns from a specific table, only ask for a the few relevant columns given the question.
    Pay attention to use only the column names that you can see in the schema description. 
    Be careful to not query for columns that do not exist.
      
    Also, pay attention to which column is in which table.
    Only use the following tables:
    {table_info}
    """),
    ("user", """
    Question:
    {input}
    """)
])


# SQL 쿼리 생성 함수
class QueryOutput(TypedDict):
    """Generated SQL query."""
    query: Annotated[str, ..., "Syntactically valid SQL query."]

def write_query(state: State):
    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "dialect": db.dialect,
            "top_k": 10,
            "table_info": db.get_table_info(),
            "input": state["question"],
        }
    )
    structured_llm = llm.with_structured_output(QueryOutput)
    result = structured_llm.invoke(prompt)
    return {"query": result["query"]}

# SQL 쿼리 실행 함수
def execute_query(state: State):
    """Execute SQL query."""
    execute_query_tool = QuerySQLDatabaseTool(db=db)
    return {"result": execute_query_tool.invoke(state["query"])}

# 답변 생성 함수
def generate_answer(state: State):
    """Answer question using retrieved information as context."""
    prompt = (
        "Given the following user question, corresponding SQL query, "
        "and SQL result, answer the user question.\n\n"
        f'Question: {state["question"]}\n'
        f'SQL Query: {state["query"]}\n'
        f'SQL Result: {state["result"]}'
    )
    response = llm.invoke(prompt)
    return {"answer": response.content}


# 상태 그래프 생성
graph_builder = StateGraph(State).add_sequence(
    [write_query, execute_query, generate_answer]
)
graph_builder.add_edge(START, "write_query")
graph = graph_builder.compile()

# 상태 그래프 시각화
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# 고유명사를 사용하는 쿼리 테스트 - 운용사
for step in graph.stream(
    {"question": "KB에서 운용하는 ETF는 모두 몇개인가요?"}, stream_mode="updates"
):
    print(step)

In [ ]:
# 고유명사를 사용하는 쿼리 테스트 - 운용사
for step in graph.stream(
    {"question": "케이비자산운용에서 운용하는 ETF는 모두 몇개인가요?"}, stream_mode="updates"
):
    print(step)

In [ ]:
# 고유명사를 사용하는 쿼리 테스트 - ETF
for step in graph.stream(
    {"question": "Dow Jones ETF는 모두 몇개인가요?"}, stream_mode="updates"
):
    print(step)

### 2) **고유명사 DB** 연동

`(1) DB 테이블에서 고유명사 추출`

In [ ]:
import ast
import re

def query_as_list(db, query):
    res = db.run(query)
    res = [el for sub in ast.literal_eval(res) for el in sub if el]
    res = [re.sub(r"\b\d+\b", "", string).strip() for string in res]
    return list(set(res))


etfs = query_as_list(db, "SELECT DISTINCT 종목명 FROM ETFs")
fund_managers = query_as_list(db, "SELECT DISTINCT 운용사 FROM ETFs")

# 결과 출력
print(f"ETF 종목 수: {len(etfs)}")
print(f"운용사 수: {len(fund_managers)}")

print("\n")

print(f"ETF 종목명: {etfs[:5]}")
print(f"운용사: {fund_managers[:5]}")

`(2) 고유명사를 벡터스토어에 저장`

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# 임베딩 모델 생성
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small") 
embeddings = OpenAIEmbeddings(model="text-embedding-3-large") # "text-embedding-3-large" 모델 비교

# 임베딩 벡터 저장소 생성
vector_store = InMemoryVectorStore(embeddings)

# ETF 종목명과 운용사를 임베딩 벡터로 변환
_ = vector_store.add_texts(etfs + fund_managers)
retriever = vector_store.as_retriever(search_kwargs={"k": 10})

In [ ]:
# 임베딩 벡터 검색 - 운용사
retriever.invoke("케이비")

In [ ]:
# 임베딩 벡터 검색 - 운용사
retriever.invoke("KB운용")

In [ ]:
# 임베딩 벡터 검색 - ETF 종목명
retriever.invoke("Dow Jones")

In [ ]:
# 임베딩 벡터 검색 - ETF 종목명
retriever.invoke("다우존스")

`(3) 검색 결과 변환하는 함수`

In [ ]:
from langchain.agents.agent_toolkits import create_retriever_tool

# 검색 프롬프트 생성
description = (
    "Use to look up values to filter on. Input is an approximate spelling "
    "of the proper noun, output is valid proper nouns. Use the noun most "
    "similar to the search."
)

# 검색 도구 생성
entity_retriever_tool = create_retriever_tool(
    retriever,
    name="search_proper_nouns",
    description=description,
)

# 검색 도구 사용
print(entity_retriever_tool.invoke("KB에서 운용하는 ETF는 모두 몇개인가요?"))

In [ ]:
print(entity_retriever_tool.invoke("Dow Jones 관련 ETF는 무엇인가요?"))

`(4) 쿼리 생성하는 사용자 프롬프트 정의`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 테이블 정보와 엔티티 관계를 포함한 프롬프트 템플릿
entity_query_template = """
Given an input question, create a syntactically correct {dialect} query to run to help find the answer. Unless the user specifies in his question a specific number of examples they wish to obtain, always limit your query to at most {top_k} results. You can order the results by a relevant column to return the most interesting examples in the database.

Never query for all the columns from a specific table, only ask for a the few relevant columns given the question.

Pay attention to use only the column names that you can see in the schema description. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

Only use the following tables:
{table_info}

Entity names and their relationships to consider:
{entity_info}

## Matching Guidelines
- Use exact matches when comparing entity names
- Check for historical name variations if available
- Apply case-sensitive matching for official names
- Handle both Korean and English entity names when present

Question: {input}
"""

# 새로운 ChatPromptTemplate 생성
query_prompt_template = ChatPromptTemplate.from_template(entity_query_template)

# 입력 변수 확인
print("Input variables:", query_prompt_template.input_variables)

In [ ]:
def write_query(state: State):
    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "dialect": db.dialect,
            "top_k": 10,
            "table_info": db.get_table_info(),
            "input": state["question"],
            "entity_info": entity_retriever_tool.invoke(state["question"]),
        }
    )
    structured_llm = llm.with_structured_output(QueryOutput)
    result = structured_llm.invoke(prompt)
    return {"query": result["query"]}


# 쿼리 실행
response = write_query({"question": "KB에서 운용하는 ETF는 모두 몇개인가요?"})

response

`(5) 상태 그래프 초기화`

In [ ]:
from langgraph.graph import START, StateGraph 

graph_builder = StateGraph(State)

graph_builder.add_node("write_query", write_query)
graph_builder.add_node("execute_query", execute_query)
graph_builder.add_node("generate_answer", generate_answer)

graph_builder.add_edge(START, "write_query")
graph_builder.add_edge("write_query", "execute_query")
graph_builder.add_edge("execute_query", "generate_answer")  
graph = graph_builder.compile()


from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# 고유명사를 사용하는 쿼리 테스트 - 운용사
for step in graph.stream(
    {"question": "KB에서 운용하는 ETF는 모두 몇개인가요?"}, stream_mode="updates"
):
    print(step)

In [ ]:
# 고유명사를 사용하는 쿼리 테스트 - 운용사
for step in graph.stream(
    {"question": "케이비에서 운용하는 ETF는 모두 몇개인가요?"}, stream_mode="updates"
):
    print(step)

In [ ]:
# 고유명사를 사용하는 쿼리 테스트 - ETF
for step in graph.stream(
    {"question": "Dow Jones ETF는 모두 몇개인가요?"}, stream_mode="updates"
):
    print(step)

---

## [실습] **고유명사 DB 구축**

- "ETFInfo" 테이블의 "기초자산" 등 다른 필드의 값에 있는 고유명사를 추출
- 추출한 고유명사를 벡터 저장소에 저장하고 검색기를 생성 
- SQL 체인 테스트

In [ ]:
# 여기에 코드를 입력하세요.